In [77]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [78]:
df=pd.read_csv('loan_data.csv')

In [79]:
df

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27.0,male,Associate,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No,1
44996,37.0,female,Associate,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,11.0,621,No,1
44997,33.0,male,Associate,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No,1
44998,29.0,male,Bachelor,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No,1


In [80]:
df.isnull().sum()

person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64

In [81]:
df.duplicated().sum()

0

In [82]:
df['person_education'].unique()

array(['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate'],
      dtype=object)

In [83]:
X=df.drop(columns='loan_status')
y=df['loan_status']

In [84]:
y.value_counts()

loan_status
0    35000
1    10000
Name: count, dtype: int64

In [85]:
Xtrain,Xtest,ytrain,ytest=train_test_split(X,y,train_size=0.8,random_state=42,stratify=y)

**OrdinalEncoder** is used only when the categories have a meaningful order.

If there is no natural ranking, use **OneHotEncoder**.

What am I trying to predict?

- Category / Label → Classification → RandomForestClassifier
- Continuous Number → Regression → RandomForestRegressor

In [86]:
Preprocessing=ColumnTransformer(
    transformers=[
        ('onehot_encoder',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),['person_gender','person_home_ownership','loan_intent','previous_loan_defaults_on_file']),
        ('ordinal_encoder',OrdinalEncoder(categories=[['High School','Associate','Bachelor','Master','Doctorate']]),['person_education'])
    ],
    remainder='passthrough'
)

In [87]:
main_pipe=Pipeline(steps=[
    ('pre',Preprocessing),
    ('under',RandomUnderSampler(random_state=42)),
    ('model',RandomForestClassifier(n_estimators=100,random_state=42))
])

In [88]:
grid_search=GridSearchCV(
    estimator=main_pipe,
    param_grid={
                'under__sampling_strategy':[0.2,0.5],
                'model__criterion':['gini','entropy'],
                'model__max_depth':[None,5,10],
                'model__min_samples_split':[2,5],
                'model__min_samples_leaf':[1,3,5],
                'model__max_features':['sqrt', 'log2']},
    cv=5,
    scoring='f1',
    n_jobs=-1

)

In [89]:
grid_search.fit(Xtrain,ytrain)

C:\Users\ANUSHA C\AppData\Roaming\Python\Python310\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\ANUSHA C\AppData\Roaming\Python\Python310\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\ANUSHA C\AppData\Roaming\Python\Python310\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\ANUSHA C\AppData\Roaming\Python\Python310\site-packages\imblearn\pipeline.py", line

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [None, 5, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 3, ...], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('onehot_encoder', ...), ('ordinal_encoder', ...)]"


The answer is:

If your goal is only to report the final performance of the model, then yes, you evaluate on the test set.

But if your goal is to determine whether the model is good, overfitting, or underfitting, you must evaluate on both the training and test sets.

In [95]:
ypred_train=grid_search.predict(Xtrain)
ypred_test=grid_search.predict(Xtest)


### Training

In [91]:
train_accuracy = accuracy_score(ytrain, ypred_train)
print("Training Accuracy :", train_accuracy)
print(classification_report(ytrain, ypred_train))


Training Accuracy : 0.9803333333333333
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     28000
           1       0.92      1.00      0.96      8000

    accuracy                           0.98     36000
   macro avg       0.96      0.99      0.97     36000
weighted avg       0.98      0.98      0.98     36000



### Testing

In [92]:
test_accuracy = accuracy_score(ytest, ypred_test)
print("Testing Accuracy :", test_accuracy)
print(classification_report(ytest, ypred_test))


Testing Accuracy : 0.9261111111111111
              precision    recall  f1-score   support

           0       0.96      0.95      0.95      7000
           1       0.82      0.85      0.84      2000

    accuracy                           0.93      9000
   macro avg       0.89      0.90      0.89      9000
weighted avg       0.93      0.93      0.93      9000

